# BAC Prep AI — Antrenament Complet pe Kaggle GPU

Acest notebook contine **ambele modele**:

| # | Model | Ce face | Parametri |
|---|-------|---------|-----------|
| **Part A** | BPE Tokenizer + Transformer Custom | Antrenat de la zero (decoder-only, ~5.4M params) | PyTorch |
| **Part B** | Qwen2.5-Math-1.5B-Instruct + LoRA | Fine-tune cu adaptoare LoRA pe 4-bit quantizat | HuggingFace + PEFT |

## Setup Kaggle
1. **Add Data** → cauta `bac-prep-data` → adauga
2. **Add Model** → cauta `qwen2.5-math-1.5b-instruct` (alekseizabirnik) → adauga
3. **GPU**: Settings → Accelerator → GPU T4 x2
4. **Internet**: Settings → Internet → On
5. **Run All**

In [ ]:
# Instalare dependinte (TOATE - pentru Part A si Part B)
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 \
    accelerate>=0.30.0 bitsandbytes>=0.46.1 datasets>=2.19.0 \
    scipy sentencepiece

In [ ]:
# Detectare automata path-uri
from pathlib import Path

def find_file(root, filename):
    for dirpath, dirnames, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None

# Date pentru Transformer custom
MERGED_PATH = find_file("/kaggle/input", "exercises_merged.json")
print(f"Merged data: {MERGED_PATH}")

# Date pentru Qwen LoRA (ChatML JSONL)
TRAIN_JSONL = find_file("/kaggle/input", "train.jsonl")
VAL_JSONL = find_file("/kaggle/input", "val.jsonl")
TEST_JSONL = find_file("/kaggle/input", "test.jsonl")
print(f"Train JSONL: {TRAIN_JSONL}")
print(f"Val JSONL:   {VAL_JSONL}")
print(f"Test JSONL:  {TEST_JSONL}")

# Incarcam datele
with open(MERGED_PATH, "r", encoding="utf-8") as f:
    all_exercises = json.load(f)
print(f"\nTotal exercitii: {len(all_exercises)}")

---
# PART A: BPE Tokenizer + Transformer Custom (de la zero)
---

## A1. BPE Tokenizer — implementat de la zero

In [ ]:
import re
from collections import Counter


class MathBPETokenizer:
    """BPE tokenizer pentru text matematic si LaTeX, implementat de la zero."""

    SPECIAL_TOKENS = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3, "<SEP>": 4}

    LATEX_TOKENS = [
        r"\frac", r"\int", r"\lim", r"\sqrt", r"\sum", r"\prod",
        r"\infty", r"\pi", r"\alpha", r"\beta", r"\gamma", r"\delta",
        r"\theta", r"\lambda", r"\sigma", r"\omega", r"\partial",
        r"\nabla", r"\rightarrow", r"\leftarrow", r"\Rightarrow",
        r"\leq", r"\geq", r"\neq", r"\approx", r"\cdot", r"\times",
        r"\div", r"\pm", r"\mp", r"\in", r"\notin", r"\subset",
        r"\cup", r"\cap", r"\forall", r"\exists",
        r"\mathbb{R}", r"\mathbb{N}", r"\mathbb{Z}", r"\mathbb{Q}", r"\mathbb{C}",
        r"\binom", r"\log", r"\ln", r"\sin", r"\cos", r"\tan",
        r"\arcsin", r"\arccos", r"\arctan",
    ]

    MATH_OPERATORS = ["+", "-", "*", "/", "=", "<", ">",
                      "(", ")", "[", "]", "{", "}", "^", "_", "|"]

    _PRE_TOKENIZE_RE = re.compile(
        r"(\\[a-zA-Z]+\{[^}]*\})"
        r"|(\\[a-zA-Z]+)"
        r"|(\d+(?:\.\d+)?)"
        r"|([a-zA-Z\u0103\u00e2\u00ee\u0219\u021b\u0102\u00c2\u00ce\u0218\u021a]+)"
        r"|([+\-*/=<>()\[\]{}^_|])"
        r"|(\s+)"
        r"|(.)")

    def __init__(self, vocab_size=8192):
        self.target_vocab_size = vocab_size
        self.vocab = {}
        self.inverse_vocab = {}
        self.merges = []

    def pre_tokenize(self, text):
        return [m.group() for m in self._PRE_TOKENIZE_RE.finditer(text) if m.group()]

    def train(self, texts, verbose=True):
        if verbose: print('[BPE] Pre-tokenizing corpus ...')
        all_pre = []
        for t in texts:
            all_pre.extend(self.pre_tokenize(t))

        protected = set(self.SPECIAL_TOKENS.keys()) | set(self.LATEX_TOKENS) | set(self.MATH_OPERATORS)
        word_freq = Counter()
        for pt in all_pre:
            word_freq[(pt,) if pt in protected else tuple(pt)] += 1

        self.vocab, idx = {}, 0
        for tok, tid in self.SPECIAL_TOKENS.items():
            self.vocab[tok] = tid; idx = max(idx, tid + 1)
        for tok in self.LATEX_TOKENS:
            if tok not in self.vocab: self.vocab[tok] = idx; idx += 1
        for tok in self.MATH_OPERATORS:
            if tok not in self.vocab: self.vocab[tok] = idx; idx += 1
        chars = set()
        for wt in word_freq:
            for s in wt: chars.add(s)
        for ch in sorted(chars):
            if ch not in self.vocab: self.vocab[ch] = idx; idx += 1

        if verbose: print(f'[BPE] Initial vocab: {len(self.vocab)}, target: {self.target_vocab_size}')

        self.merges = []
        num_merges = self.target_vocab_size - len(self.vocab)
        for i in range(max(0, num_merges)):
            pair_counts = Counter()
            for wt, freq in word_freq.items():
                for j in range(len(wt) - 1):
                    pair_counts[(wt[j], wt[j+1])] += freq
            if not pair_counts: break
            best = pair_counts.most_common(1)[0][0]
            merged = best[0] + best[1]
            self.merges.append(best)
            if merged not in self.vocab: self.vocab[merged] = idx; idx += 1
            new_wf = Counter()
            for wt, freq in word_freq.items():
                new_wf[self._apply_merge(wt, best)] += freq
            word_freq = new_wf
            if verbose and (i+1) % 500 == 0:
                print(f'[BPE] Merge {i+1}/{num_merges}: {best[0]!r}+{best[1]!r} -> {merged!r}')

        self.inverse_vocab = {v: k for k, v in self.vocab.items()}
        if verbose: print(f'[BPE] Done. Final vocab: {len(self.vocab)}')

    def encode(self, text):
        protected = set(self.SPECIAL_TOKENS) | set(self.LATEX_TOKENS) | set(self.MATH_OPERATORS)
        ids = []
        for pt in self.pre_tokenize(text):
            if pt in protected:
                ids.append(self.vocab.get(pt, 1))
            else:
                symbols = list(pt)
                for left, right in self.merges:
                    symbols = self._apply_merge_list(symbols, left, right)
                ids.extend(self.vocab.get(s, 1) for s in symbols)
        return ids

    def decode(self, ids, strip_special=True):
        special_ids = set(self.SPECIAL_TOKENS.values()) if strip_special else set()
        tokens = [self.inverse_vocab.get(i, '') for i in ids if not (strip_special and i in special_ids)]
        text = ''.join(tokens)
        if strip_special:
            for st in self.SPECIAL_TOKENS: text = text.replace(st, '')
        return text.strip()

    def save(self, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'vocab': self.vocab, 'merges': [list(p) for p in self.merges]}, f, ensure_ascii=False, indent=2)

    def load(self, path):
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        self.vocab = d['vocab']; self.merges = [tuple(p) for p in d['merges']]
        self.inverse_vocab = {v: k for k, v in self.vocab.items()}

    def __len__(self): return len(self.vocab)

    @staticmethod
    def _apply_merge(word, pair):
        new, i = [], 0
        while i < len(word):
            if i < len(word)-1 and word[i]==pair[0] and word[i+1]==pair[1]:
                new.append(pair[0]+pair[1]); i += 2
            else: new.append(word[i]); i += 1
        return tuple(new)

    @staticmethod
    def _apply_merge_list(symbols, left, right):
        merged, i = [], 0
        while i < len(symbols):
            if i < len(symbols)-1 and symbols[i]==left and symbols[i+1]==right:
                merged.append(left+right); i += 2
            else: merged.append(symbols[i]); i += 1
        return merged


print('MathBPETokenizer definit')

In [ ]:
# Extragem textele din exercitii si antrenam tokenizer-ul
def extract_texts(data):
    texts = []
    def _recurse(obj):
        if isinstance(obj, str) and obj.strip(): texts.append(obj.strip())
        elif isinstance(obj, list):
            for item in obj: _recurse(item)
        elif isinstance(obj, dict):
            for v in obj.values(): _recurse(v)
    _recurse(data)
    return texts

texts = extract_texts(all_exercises)
print(f'Extracted {len(texts)} text fragments')

tokenizer_bpe = MathBPETokenizer(vocab_size=8192)
tokenizer_bpe.train(texts, verbose=True)

TOKENIZER_SAVE = '/kaggle/working/math_bpe.json'
tokenizer_bpe.save(TOKENIZER_SAVE)
print(f'\nTokenizer salvat: {TOKENIZER_SAVE}')
print(f'Vocab size: {len(tokenizer_bpe)}')

# Test rapid
for s in ['Rezolvati ecuatia: x^2 + 3x - 4 = 0', r'\frac{1}{2} + \frac{3}{4}']:
    ids = tokenizer_bpe.encode(s)
    dec = tokenizer_bpe.decode(ids)
    print(f'  [{"OK" if dec==s else "!"}] {s[:50]} -> {len(ids)} tokens')

## A2. Transformer Decoder-Only — implementat de la zero (~5.4M params)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


@dataclass
class MathTransformerConfig:
    vocab_size: int = 8192
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 6
    d_ff: int = 1024
    max_seq_len: int = 512
    dropout: float = 0.1
    pad_idx: int = 0
    bos_idx: int = 2
    eos_idx: int = 3
    sep_idx: int = 4


def create_causal_mask(seq_len, device):
    """Masca cauzala triunghiulara — fiecare token vede doar tokenii anteriori."""
    return torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1).unsqueeze(0).unsqueeze(0)


class MultiHeadSelfAttention(nn.Module):
    """Multi-Head Self-Attention implementat manual (fara nn.MultiheadAttention)."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads, self.d_k = d_model, n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.output_dropout = nn.Dropout(dropout)

    def forward(self, x, causal_mask=None):
        B, S, _ = x.shape
        Q = self.W_q(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask, float('-inf'))
        weights = self.attn_dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(weights, V).transpose(1, 2).contiguous().view(B, S, self.d_model)
        return self.output_dropout(self.W_o(out))


class FeedForward(nn.Module):
    """Position-wise FFN cu GELU: FFN(x) = W2 * GELU(W1 * x + b1) + b2"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(d_model, d_ff)
        self.W2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.W2(self.dropout(F.gelu(self.W1(x))))


class SinusoidalPositionalEncoding(nn.Module):
    """Codificare pozitionala sinusoidala (Vaswani et al., 2017)."""
    def __init__(self, d_model, max_len=2048, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class TransformerBlock(nn.Module):
    """Bloc Transformer Pre-Norm (stilul GPT-2)."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, causal_mask=None):
        x = x + self.dropout1(self.attention(self.norm1(x), causal_mask))
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x


class MathTransformer(nn.Module):
    """
    Transformer autoregresiv (decoder-only) pentru rezolvarea exercitiilor BAC.
    Arhitectura similara cu GPT-2, dar mai mica (~5.4M params).
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_idx)
        self.positional_encoding = SinusoidalPositionalEncoding(config.d_model, config.max_seq_len, config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config.d_model, config.n_heads, config.d_ff, config.dropout) for _ in range(config.n_layers)])
        self.final_norm = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self._init_weights()
        n_params = sum(p.numel() for p in self.parameters())
        print(f'MathTransformer: {n_params:,} parameters')

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0, 0.02)
                if m.padding_idx is not None:
                    with torch.no_grad(): m.weight[m.padding_idx].fill_(0)

    def forward(self, input_ids):
        B, S = input_ids.shape
        x = self.token_embedding(input_ids) * math.sqrt(self.config.d_model)
        x = self.positional_encoding(x)
        mask = create_causal_mask(S, input_ids.device)
        for block in self.blocks:
            x = block(x, mask)
        return self.lm_head(self.final_norm(x))

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=200, temperature=0.7, top_k=50, eos_idx=None):
        self.eval()
        if eos_idx is None: eos_idx = self.config.eos_idx
        generated = input_ids.clone()
        finished = torch.zeros(input_ids.size(0), dtype=torch.bool, device=input_ids.device)
        for _ in range(max_new_tokens):
            ctx = generated[:, -self.config.max_seq_len:] if generated.size(1) > self.config.max_seq_len else generated
            logits = self.forward(ctx)[:, -1, :]
            if temperature != 1.0: logits = logits / temperature
            if top_k > 0:
                topk_v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < topk_v[:, -1:], float('-inf'))
            probs = torch.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            next_tok = next_tok.masked_fill(finished.unsqueeze(1), self.config.pad_idx)
            generated = torch.cat([generated, next_tok], dim=1)
            finished = finished | (next_tok.squeeze(1) == eos_idx)
            if finished.all(): break
        return generated


print('Clasele modelului definite')

## A3. Dataset + Antrenament Transformer

In [ ]:
from torch.utils.data import Dataset, DataLoader


class MathDataset(Dataset):
    """Dataset pentru next-token prediction: <BOS> question <SEP> answer <SEP> steps... <EOS>"""
    BOS_ID, EOS_ID, SEP_ID = 2, 3, 4

    def __init__(self, exercises, tokenizer, max_len=256):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.sequences = [self._build(ex) for ex in exercises]

    def _build(self, item):
        q = item.get('question', '')
        a = str(item.get('answer', ''))
        steps = item.get('solution_steps', item.get('steps', []))
        seq = [self.BOS_ID] + self.tokenizer.encode(q) + [self.SEP_ID] + self.tokenizer.encode(a)
        for s in steps:
            seq += [self.SEP_ID] + self.tokenizer.encode(str(s))
        seq.append(self.EOS_ID)
        seq = seq[:self.max_len]
        if seq[-1] != self.EOS_ID: seq[-1] = self.EOS_ID
        return seq

    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx): return torch.tensor(self.sequences[idx], dtype=torch.long)


def collate_fn(batch, pad_idx=0):
    return nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=pad_idx)


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, lr_max=3e-4):
        self.optimizer, self.warmup_steps = optimizer, warmup_steps
        self.total_steps, self.lr_max, self._step = total_steps, lr_max, 0

    def step(self):
        self._step += 1
        lr = self._compute_lr()
        for pg in self.optimizer.param_groups: pg['lr'] = lr

    def _compute_lr(self):
        if self._step <= self.warmup_steps:
            return self.lr_max * (self._step / max(self.warmup_steps, 1))
        progress = (self._step - self.warmup_steps) / max(self.total_steps - self.warmup_steps, 1)
        return self.lr_max * 0.5 * (1.0 + math.cos(math.pi * progress))

    def get_last_lr(self): return self._compute_lr()


# Split data
random.seed(42)
indices = list(range(len(all_exercises)))
random.shuffle(indices)
n_train = int(0.8 * len(all_exercises))
n_val = int(0.1 * len(all_exercises))
train_data_tf = [all_exercises[i] for i in indices[:n_train]]
val_data_tf = [all_exercises[i] for i in indices[n_train:n_train+n_val]]
test_data_tf = [all_exercises[i] for i in indices[n_train+n_val:]]

MAX_SEQ_LEN = 256
train_ds = MathDataset(train_data_tf, tokenizer_bpe, MAX_SEQ_LEN)
val_ds = MathDataset(val_data_tf, tokenizer_bpe, MAX_SEQ_LEN)
test_ds = MathDataset(test_data_tf, tokenizer_bpe, MAX_SEQ_LEN)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
avg_len = sum(len(s) for s in train_ds.sequences) / len(train_ds.sequences)
print(f'Lungime medie secventa: {avg_len:.1f} tokens')

In [ ]:
# Hyperparametri si model
BATCH_SIZE = 16
NUM_EPOCHS = 50
LR_MAX = 3e-4
GRAD_CLIP = 1.0

device = torch.device('cuda')
vocab_size = len(tokenizer_bpe)

config_tf = MathTransformerConfig(
    vocab_size=vocab_size, d_model=256, n_heads=8, n_layers=6,
    d_ff=1024, max_seq_len=MAX_SEQ_LEN, dropout=0.1,
)

model_tf = MathTransformer(config_tf).to(device)

pad_idx = config_tf.pad_idx
_collate = lambda batch: collate_fn(batch, pad_idx)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=_collate)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=_collate)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=_collate)

optimizer_tf = torch.optim.AdamW(model_tf.parameters(), lr=LR_MAX, betas=(0.9, 0.95), weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

total_steps = NUM_EPOCHS * len(train_loader)
warmup_steps = int(0.10 * total_steps)
scheduler = WarmupCosineScheduler(optimizer_tf, warmup_steps, total_steps, LR_MAX)
print(f'Total steps: {total_steps}, Warmup: {warmup_steps}')

In [ ]:
# Antrenament
best_val_loss = float('inf')
history_tf = []
CHECKPOINT_DIR = '/kaggle/working/transformer_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

print(f'\n{"="*65}')
print(f'  Antrenament Transformer Custom: {NUM_EPOCHS} epoci pe {device}')
print(f'{"="*65}\n')

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # Train
    model_tf.train()
    train_loss, n_batches = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device)
        input_ids, labels = batch[:, :-1], batch[:, 1:]
        optimizer_tf.zero_grad()
        logits = model_tf(input_ids)
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model_tf.parameters(), GRAD_CLIP)
        optimizer_tf.step()
        scheduler.step()
        train_loss += loss.item()
        n_batches += 1
    train_loss /= max(n_batches, 1)

    # Validate
    model_tf.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            input_ids, labels = batch[:, :-1], batch[:, 1:]
            logits = model_tf(input_ids)
            val_loss += criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1)).item()
            n_val += 1
    val_loss /= max(n_val, 1)

    elapsed = time.time() - t0
    lr = scheduler.get_last_lr()
    print(f'  Epoca {epoch:3d}/{NUM_EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f} | lr={lr:.2e} | {elapsed:.1f}s')

    history_tf.append({'epoch': epoch, 'train_loss': round(train_loss, 6), 'val_loss': round(val_loss, 6)})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch, 'model_state_dict': model_tf.state_dict(),
            'optimizer_state_dict': optimizer_tf.state_dict(),
            'config': config_tf, 'val_loss': val_loss, 'vocab_size': vocab_size,
        }, best_model_path)
        print(f'    -> Best model salvat (val_loss={val_loss:.4f})')

print(f'\nBest val_loss: {best_val_loss:.4f}')

In [ ]:
# Evaluare pe test set
model_tf.eval()
test_loss, n_test = 0.0, 0
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        input_ids, labels = batch[:, :-1], batch[:, 1:]
        logits = model_tf(input_ids)
        test_loss += criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1)).item()
        n_test += 1
test_loss /= max(n_test, 1)
print(f'Test loss: {test_loss:.4f}')

In [ ]:
# Generare demonstrativa
demos = [
    'Rezolva ecuatia: 2x + 3 = 7',
    'Calculati derivata functiei f(x) = x^3',
    'Calculati det[[3,1],[2,4]]',
]

for prompt in demos:
    prompt_ids = [config_tf.bos_idx] + tokenizer_bpe.encode(prompt) + [config_tf.sep_idx]
    inp = torch.tensor([prompt_ids], dtype=torch.long).to(device)
    out = model_tf.generate(inp, max_new_tokens=100, temperature=0.5, top_k=30)
    gen_ids = out[0].tolist()[len(prompt_ids):]
    gen_text = tokenizer_bpe.decode(gen_ids)
    print(f'\nQ: {prompt}')
    print(f'A: {gen_text}')

In [ ]:
# Curbe de antrenament Part A
import matplotlib.pyplot as plt

epochs_list = [h['epoch'] for h in history_tf]
train_losses = [h['train_loss'] for h in history_tf]
val_losses = [h['val_loss'] for h in history_tf]

plt.figure(figsize=(10, 5))
plt.plot(epochs_list, train_losses, label='Train Loss')
plt.plot(epochs_list, val_losses, label='Val Loss')
plt.xlabel('Epoca'); plt.ylabel('Loss'); plt.title('MathTransformer Custom — Training Curves')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig('/kaggle/working/transformer_training_curves.png', dpi=150)
plt.show()

# Salvam history
with open('/kaggle/working/transformer_training_history.json', 'w') as f:
    json.dump(history_tf, f, indent=2)

# Eliberam GPU pentru Part B
del model_tf, optimizer_tf, scheduler, train_loader, val_loader, test_loader
torch.cuda.empty_cache()
print(f'GPU VRAM eliberat. Trecem la Part B.')

---
# PART B: Qwen2.5-Math-1.5B + LoRA Fine-tuning

**NU antrenam de la zero** — luam modelul pre-antrenat Qwen, il cuantizam pe 4-bit, si aplicam adaptoare LoRA.

---

## B1. Instalare dependinte

In [ ]:
# Dependintele au fost deja instalate la inceputul notebook-ului
print('OK — dependinte deja instalate')

## B2. Incarcare date ChatML

In [ ]:
def load_jsonl(path):
    samples = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples

train_data_lora = load_jsonl(TRAIN_JSONL)
val_data_lora = load_jsonl(VAL_JSONL)
test_data_lora = load_jsonl(TEST_JSONL)

print(f'Train: {len(train_data_lora)} | Val: {len(val_data_lora)} | Test: {len(test_data_lora)}')
print(f'\nSample:\n{train_data_lora[0]["text"][:300]}...')

## B3. Incarcare Qwen2.5-Math-1.5B-Instruct + Quantizare 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Modelul Qwen adaugat ca Kaggle Model
MODEL_NAME = '/kaggle/input/models/alekseizabirnik/qwen2.5-math-1.5b-instruct/transformers/default/1'
print(f'Model path: {MODEL_NAME}')

# Quantizare 4-bit NF4 — reduce VRAM de la ~6GB la ~1.5GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Incarc modelul...')
model_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True, local_files_only=True,
)

tokenizer_qwen = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side='right', local_files_only=True,
)

if tokenizer_qwen.pad_token is None:
    tokenizer_qwen.pad_token = tokenizer_qwen.eos_token
    model_qwen.config.pad_token_id = tokenizer_qwen.eos_token_id

print(f'Model incarcat! Parametri: {model_qwen.num_parameters():,}')
print(f'VRAM folosit: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## B4. Aplicare adaptoare LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model_qwen = prepare_model_for_kbit_training(model_qwen)

lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # Scaling factor (2x rank)
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model_qwen = get_peft_model(model_qwen, lora_config)
model_qwen.print_trainable_parameters()

## B5. Pregatire Dataset

In [ ]:
from datasets import Dataset as HFDataset

MAX_SEQ_LEN_QWEN = 512

def tokenize_sample(sample):
    result = tokenizer_qwen(
        sample['text'], truncation=True,
        max_length=MAX_SEQ_LEN_QWEN, padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

train_dataset_lora = HFDataset.from_list(train_data_lora).map(tokenize_sample, remove_columns=['text'])
val_dataset_lora = HFDataset.from_list(val_data_lora).map(tokenize_sample, remove_columns=['text'])

train_dataset_lora.set_format('torch')
val_dataset_lora.set_format('torch')

print(f'Train: {len(train_dataset_lora)} | Val: {len(val_dataset_lora)}')

## B6. Antrenament LoRA

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR_LORA = '/kaggle/working/qwen-bac-lora'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR_LORA,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    weight_decay=0.01,
    fp16=False, bf16=False,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=10,
    logging_first_step=True,
    report_to='none',
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    max_grad_norm=1.0,
    seed=42,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model_qwen,
    args=training_args,
    train_dataset=train_dataset_lora,
    eval_dataset=val_dataset_lora,
)

print('Incep antrenamentul LoRA...')
print(f'  Effective batch size: {4 * 4}')
print(f'  Total steps: ~{len(train_dataset_lora) // 16 * 3}')

In [ ]:
train_result = trainer.train()

print(f"\n{'='*60}")
print(f"LoRA Training complet!")
print(f"  Train loss: {train_result.metrics['train_loss']:.4f}")
print(f"  Runtime:    {train_result.metrics['train_runtime']:.0f}s")
print(f"{'='*60}")

## B7. Evaluare pe Test Set

In [ ]:
test_dataset_lora = HFDataset.from_list(test_data_lora).map(tokenize_sample, remove_columns=['text'])
test_results = trainer.evaluate(eval_dataset=test_dataset_lora)
print(f'Test loss: {test_results["eval_loss"]:.4f}')
print(f'Test perplexity: {torch.exp(torch.tensor(test_results["eval_loss"])):.2f}')

## B8. Salvare adapter LoRA

In [ ]:
ADAPTER_PATH = '/kaggle/working/qwen-bac-lora/best_adapter'
trainer.save_model(ADAPTER_PATH)
tokenizer_qwen.save_pretrained(ADAPTER_PATH)

total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_PATH, f))
    for f in os.listdir(ADAPTER_PATH)
    if os.path.isfile(os.path.join(ADAPTER_PATH, f))
)
print(f'Adapter salvat: {ADAPTER_PATH}')
print(f'Dimensiune: {total_size / 1e6:.1f} MB')

## B9. Test inferenta

In [ ]:
test_questions = [
    'Rezolva ecuatia: 2x + 3 = 7',
    'Calculati derivata functiei f(x) = x^3 - 6x^2 + 9x + 1',
    'Calculati limita: lim(x->inf) (x^2 + 3x) / (2x^2 - 1)',
    'Calculati det[[3,1],[2,4]]',
]

SYSTEM_PROMPT = (
    'Ești un asistent de matematică specializat pe exerciții BAC. '
    'Rezolvă exercițiul pas cu pas, explicând fiecare etapă.'
)

model_qwen.eval()
for q in test_questions:
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n'
        f'<|im_start|>user\n{q}\n<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer_qwen(prompt, return_tensors='pt').to(model_qwen.device)
    with torch.no_grad():
        outputs = model_qwen.generate(
            **inputs, max_new_tokens=256, temperature=0.1,
            top_k=30, do_sample=True,
            pad_token_id=tokenizer_qwen.pad_token_id,
        )
    response = tokenizer_qwen.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    if '<|im_end|>' in response:
        response = response[:response.index('<|im_end|>')]
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'A: {response.strip()}')

In [ ]:
# Curbe antrenament LoRA
log_history = trainer.state.log_history
train_steps = [h['step'] for h in log_history if 'loss' in h]
train_losses_lora = [h['loss'] for h in log_history if 'loss' in h]
eval_steps_log = [h['step'] for h in log_history if 'eval_loss' in h]
eval_losses_lora = [h['eval_loss'] for h in log_history if 'eval_loss' in h]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses_lora, label='Train Loss', alpha=0.7)
plt.plot(eval_steps_log, eval_losses_lora, 'ro-', label='Val Loss', markersize=4)
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Qwen2.5-Math LoRA Fine-tuning — Training Curves')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig('/kaggle/working/lora_training_curves.png', dpi=150)
plt.show()

<cell_type>markdown</cell_type>---
# Sumar + Push to HuggingFace
---

In [ ]:
# ============================================================
# PUSH ADAPTER LoRA PE HUGGING FACE
# ============================================================
# Astfel modelul e salvat permanent si nu se pierde din /kaggle/working

!pip install -q huggingface_hub

from huggingface_hub import login, HfApi
import os

# ==============================
# CONFIGUREAZA AICI
# ==============================
HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXX'  # <-- Pune tokenul tau HuggingFace (Settings -> Access Tokens -> New token, Write)
HF_REPO = 'USERNAME/bac-math-qwen-lora'  # <-- Pune username-ul tau HuggingFace

login(token=HF_TOKEN)
api = HfApi()

# Creeaza repo-ul (daca nu exista)
try:
    api.create_repo(repo_id=HF_REPO, repo_type='model', private=True, exist_ok=True)
    print(f'Repo creat/exista: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'Repo error: {e}')

# Upload adaptorul LoRA
print('\nUpload adapter LoRA...')
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=HF_REPO,
    path_in_repo='best_adapter',
    commit_message='Upload LoRA adapter antrenat pe exercitii BAC',
)
print(f'Adapter uploadat: https://huggingface.co/{HF_REPO}/tree/main/best_adapter')

# Upload tokenizer BPE custom
print('\nUpload tokenizer BPE...')
api.upload_file(
    path_or_fileobj='/kaggle/working/math_bpe.json',
    path_in_repo='math_bpe.json',
    repo_id=HF_REPO,
    commit_message='Upload BPE tokenizer custom',
)

# Upload checkpoint Transformer custom
print('\nUpload checkpoint Transformer...')
api.upload_file(
    path_or_fileobj=best_model_path,
    path_in_repo='transformer_checkpoints/best_model.pt',
    repo_id=HF_REPO,
    commit_message='Upload MathTransformer custom checkpoint',
)

# Upload curbele de antrenament
for fname in ['transformer_training_curves.png', 'lora_training_curves.png', 'transformer_training_history.json']:
    fpath = f'/kaggle/working/{fname}'
    if os.path.exists(fpath):
        api.upload_file(path_or_fileobj=fpath, path_in_repo=fname, repo_id=HF_REPO)

print(f'\n{"="*60}')
print(f'  TOTUL UPLOADAT PE HUGGING FACE!')
print(f'  https://huggingface.co/{HF_REPO}')
print(f'{"="*60}')

In [ ]:
print('='*60)
print('  SUMAR ANTRENAMENT')
print('='*60)
print(f'\n  PART A: Transformer Custom (de la zero)')
print(f'    Parametri:   ~5.4M')
print(f'    Best val_loss: {best_val_loss:.4f}')
print(f'    Test loss:     {test_loss:.4f}')
print(f'    Checkpoint:    {best_model_path}')
print(f'\n  PART B: Qwen2.5-Math-1.5B + LoRA')
print(f'    Parametri:     1.5B (4-bit quantizat)')
print(f'    LoRA rank:     16, alpha: 32')
print(f'    Train loss:    {train_result.metrics["train_loss"]:.4f}')
print(f'    Test loss:     {test_results["eval_loss"]:.4f}')
print(f'    Test ppl:      {torch.exp(torch.tensor(test_results["eval_loss"])):.2f}')
print(f'    Adapter:       {ADAPTER_PATH}')
print(f'\n  HUGGING FACE: https://huggingface.co/{HF_REPO}')
print('='*60)

# Backup: comprima totul pentru download manual
!cd /kaggle/working && tar -czf bac_models.tar.gz \
    transformer_checkpoints/best_model.pt \
    math_bpe.json \
    qwen-bac-lora/best_adapter/ \
    transformer_training_curves.png \
    lora_training_curves.png \
    transformer_training_history.json

size = os.path.getsize('/kaggle/working/bac_models.tar.gz') / 1e6
print(f'\nBackup: bac_models.tar.gz ({size:.1f} MB)')